# Library

In [1]:
import csv
import time
import json
import os
import asyncio

from search_on_company_site import on_company_site_search
from search_on_jpx import jpx_governance_search
from search_on_nikkei import nikkei_governance_search

from validator_llm import SearchReportValidator
from searcher_searxng import SearXNGSearch
from search_combine import SearchReportCombine
from automation_bot import AutomationBot, run_bot

validator = SearchReportValidator()
searcher = SearXNGSearch()

/Users/hoan.hk/Desktop/Works/Stock/venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/hoan.hk/Desktop/Works/Stock/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

In [2]:
company_info = {
    "6920": {"name": "Lasertec Corporation", "site": "https://www.lasertec.co.jp/"},
    "4063": {"name": "Shin-Etsu Chemical Co., Ltd.", "site": "https://www.shinetsu.co.jp/"},
    "9104": {"name": "Mitsui O.S.K. Lines, Ltd.", "site": "https://www.mol.co.jp/"},
    "2432": {"name": "DeNA Co., Ltd.", "site": "https://dena.com/intl/"},
    "9697": {"name": "CAPCOM CO., LTD.", "site": "https://www.capcom.co.jp/"},
    "5108": {"name": "BRIDGESTONE CORPORATION", "site": "https://www.bridgestone.co.jp/"},
    "4568": {"name": "DAIICHI SANKYO COMPANY, LIMITED", "site": "https://www.daiichisankyo.co.jp/"},
    "1812": {"name": "KAJIMA CORPORATION", "site": "https://www.kajima.co.jp/"},
    "5411": {"name": "JFE Holdings, Inc.", "site": "https://www.jfe-holdings.co.jp/"},
    "7974": {"name": "Nintendo Co., Ltd.", "site": "https://www.nintendo.co.jp/"},
    "7011": {"name": "Mitsubishi Heavy Industries, Ltd.", "site": "https://www.mhi.com/"},
    "8001": {"name": "ITOCHU Corporation", "site": "https://www.itochu.co.jp/"},
    "6952": {"name": "CASIO COMPUTER CO., LTD.", "site": "https://www.casio.co.jp/"},
    "4393": {"name": "Bank of Innovation, Inc.", "site": "https://www.boi.jp/"},
    "8306": {"name": "Mitsubishi UFJ Financial Group, Inc.", "site": "https://www.mufg.jp/"},
    "6857": {"name": "ADVANTEST CORPORATION", "site": "https://www.advantest.com/"},
    "7203": {"name": "TOYOTA MOTOR CORPORATION", "site": "https://www.global.toyota/"},
    "9983": {"name": "FAST RETAILING CO., LTD.", "site": "https://www.fastretailing.com/"},
    "8801": {"name": "Mitsui Fudosan Co., Ltd.", "site": "https://www.mitsuifudosan.co.jp/"},
    "9984": {"name": "SoftBank Group Corp.", "site": "https://www.group.softbank/"},
    "3635": {"name": "KOEI TECMO HOLDINGS CO., LTD.", "site": "https://www.koeitecmo.co.jp/"},
    "4816": {"name": "TOEI ANIMATION CO., LTD.", "site": "https://www.toei-anim.co.jp/"},
    "5020": {"name": "ENEOS Holdings, Inc.", "site": "https://www.hd.eneos.co.jp/"},
    "8058": {"name": "Mitsubishi Corporation", "site": "https://www.mitsubishicorp.com/"},
    "4689": {"name": "LY Corporation", "site": "https://www.lycorp.co.jp/"},
}

# A/ Intergrated report

### 1 - 🌐 Search On Company Site

#### Flow Diagram

```
┌───────────────────────────────────────────────────────────────────────┐
│   Company Website Toyota                                              │
│  Search: "site:toyota:global integrated reports filetype:pdf"         │
└───────────────────────────────────────────────────────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │   SearXNG Search      │
        │  (meta-search engine) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Validate with LLM    │
        │  (best report filter) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Extract & Save CSV   │
        │  (URL, Date, Source)  │
        └───────────────────────┘
```


In [ ]:
on_company_site_search(
    searcher,
    validator,
    company_info=company_info,
    output_file="integrated_reports_on_company_site.csv",
    search_keyword="integrated report",
    result_label="Integrated"
)

# B/ Corporate governance report

### 1 - 🌐 Search On Company Site

#### Flow Diagram

```
┌───────────────────────────────────────────────────────────────────────────┐
│   Company Website Toyota                                                  │
│  Search: "site:toyota:global corporate governance reports filetype:pdf"   │
└───────────────────────────────────────────────────────────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │   SearXNG Search      │
        │  (meta-search engine) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Validate with LLM    │
        │  (best report filter) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Extract & Save CSV   │
        │  (URL, Date, Source)  │
        └───────────────────────┘
```


In [ ]:
on_company_site_search(
    searcher,
    validator,
    company_info=company_info,
    output_file="corporate_governance_reports_on_company_site.csv",
    search_keyword="corporate governance report",
    result_label="Governance"
)

### 2 - 🗞 Search On Nikkei

#### Flow Diagram

```
┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   ------Search on Nikkei site------                                                                                                  │
│  Search: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 {name} filetype:pdf                    │
│  Example: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 TOYOTA MOTOR CORPORATION filetype:pdf │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │   SearXNG Search      │
        │  (meta-search engine) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Validate with LLM    │
        │  (best report filter) │
        └───────────────────────┘
                    │
                    ▼
        ┌───────────────────────┐
        │  Extract & Save CSV   │
        │  (URL, Date, Source)  │
        └───────────────────────┘
```


In [ ]:
nikkei_governance_search(
    searcher,
    validator,
    company_info,
    output_file="nikkei_governance_best_results.csv",
    delay=10
)

### 3 - 🏛 Search On JPX

#### Flow Diagram

```
┌──────────────────────────────────────────────────────────────┐
│  Japan Exchange Group (JPX)                                  │
│  Listed Company Search Database                              │
│                                                              │
│  URL: https://www2.jpx.co.jp/tseHpFront/JJK020030Action.do   │
└──────────────────────────────────────────────────────────────┘
                │
                ▼
    ┌──────────────────────────┐
    │  Search by Stock Code    │──┐ 
    │  (Direct lookup)         │  │
    └──────────────────────────┘  │
                │                 │ - [Playwright]
                ▼                 │
    ┌──────────────────────────┐  │
    │  Click Corporate         │  │
    │  Governance Tab          │──┘
    └──────────────────────────┘
                │                 
                ▼                 
    ┌──────────────────────────┐
    │  Extract Japanese Table  │
    │  (Latest governance data)│
    └──────────────────────────┘
                │
                ▼
    ┌──────────────────────────┐
    │  Parse PDF Link & Date   │
    │  (Official document)     │
    └──────────────────────────┘
                │
                ▼
    ┌──────────────────────────┐
    │  Save to CSV             │
    │  (Date, URL, Source)     │
    └──────────────────────────┘
```


In [ ]:
await jpx_governance_search(
    company_info,
    output_file="jpx_governance.csv",
    headless = True
)

### 4 - 🧩 Search Combine

#### Flow Diagram

```
┌─────────────────────────────────────────┐
│  Multiple Search Sources Combined       │
└─────────────────────────────────────────┘
    │                  │                  │
    ▼                  ▼                  ▼
┌─────────┐      ┌──────────┐      ┌──────────┐
│Company  │      │ Nikkei   │      │   JPX    │
│ Site    │      │ Site     │      │  Site    │
└─────────┘      └──────────┘      └──────────┘
    │                  │                  │
    └──────────────────┼──────────────────┘
                       │
                       ▼
        ┌─────────────────────────────┐
        │  Parse & Normalize Results  │
        │  (Date, URL, Source)        │
        └─────────────────────────────┘
                       │
                       ▼
        ┌─────────────────────────────┐
        │  Compare & Select Latest    │
        │  (Most recent date)         │
        └─────────────────────────────┘
                       │
                       ▼
        ┌─────────────────────────────┐
        │  Save Best Result to CSV    │
        │  (Highest quality source)   │
        └─────────────────────────────┘
```


In [ ]:
governance_combine_search = SearchReportCombine()
await governance_combine_search.process_companies(company_info, "combine_governance.csv")

# C/ Automation bot

In [ ]:
auto_bot = AutomationBot(headless=True, report_type="Integrated Report")

await run_bot(
    auto_bot=auto_bot,
    company_info=company_info,
    output_file="integrated_report_results_bot.csv"
)

✓ Graph saved successfully: graph.png

🔎 Processing Lasertec Corporation (6920)

Crawling: https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/ir/data/'}
📍 Next URL: https://www.lasertec.co.jp/ir/data/

Crawling: https://www.lasertec.co.jp/ir/data/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/ir/data/'}
📍 Next URL: https://www.lasertec.co.jp/ir/data/

Crawling: https://www.lasertec.co.jp/ir/data/
🔗 Extracted 125 links from https://www.lasertec.co.jp/ir/data/
🔗 Extracted 125 links from https://www.lasertec.co.jp/ir/data/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/ir/data/integrated_report.html'}
📍 Next URL: https://www.lasertec.co.jp/ir/data/integrated_report.html

Crawling: https://www.lasertec.co.jp/ir/data/integrated_report.html
🤖 LLM: {'next_step': 'continue_find', 'url

In [3]:
# company_info = {
#     "6920": {"name": "Lasertec Corporation", "site": "https://www.lasertec.co.jp/"},
# }

# auto_bot = AutomationBot(headless=True, report_type="Corporate Governance Report")

# await run_bot(
#     auto_bot=auto_bot,
#     company_info=company_info,
#     output_file="test.csv"
# )

✓ Graph saved successfully: graph.png

🔎 Processing Lasertec Corporation (6920)

Crawling: https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/sustainability/governance.html'}
📍 Next URL: https://www.lasertec.co.jp/sustainability/governance.html

Crawling: https://www.lasertec.co.jp/sustainability/governance.html
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/sustainability/governance.html'}
📍 Next URL: https://www.lasertec.co.jp/sustainability/governance.html

Crawling: https://www.lasertec.co.jp/sustainability/governance.html
🔗 Extracted 105 links from https://www.lasertec.co.jp/sustainability/governance.html
🔗 Extracted 105 links from https://www.lasertec.co.jp/sustainability/governance.html
🤖 LLM: {'next_step': 'END', 'url': 'https://ssl4.eir-parts.net/doc/6920/tdnet/2691142/00.pdf'}
✅ Found PDF: htt